In [15]:
from pathlib import Path
import json
import os
import pickle

import numpy as np
import pandas as pd
from huggingface_hub import InferenceClient
from IPython.display import display
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline


In [16]:
train_path = Path('../data/nlp/prepared/model1_train.csv')
val_path = Path('../data/nlp/prepared/model1_val.csv')
test_path = Path('../data/nlp/prepared/model1_test.csv')

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print('train:', train_df.shape, 'val:', val_df.shape, 'test:', test_df.shape)
display(train_df[['clean_text', 'target_price_tnd']].head(3))


train: (3214, 9) val: (401, 9) test: (403, 9)


,clean_text,target_price_tnd
0,appartement à vendre à sousse sousse surface 1...,315000.0
1,appartement à vendre à l aouina tunis pièces 1...,208000.0
2,appartement disponible à medina jedida ben aro...,200000.0


In [17]:
def get_xy(df: pd.DataFrame):
    X = df['clean_text'].fillna('').astype(str)
    y = pd.to_numeric(df['target_price_tnd'], errors='coerce')
    mask = X.str.len().gt(0) & y.notna() & y.gt(0)
    return X[mask].reset_index(drop=True), y[mask].reset_index(drop=True)


def regression_metrics(y_true, pred, model_name: str, split_name: str):
    return {
        'model': model_name,
        'split': split_name,
        'mae': float(mean_absolute_error(y_true, pred)),
        'rmse': float(np.sqrt(mean_squared_error(y_true, pred))),
        'r2': float(r2_score(y_true, pred)),
    }


def evaluate_regression_model(model, model_name: str, X, y, split_name: str):
    pred = model.predict(X)
    return regression_metrics(y, pred, model_name=model_name, split_name=split_name), pred


X_train, y_train = get_xy(train_df)
X_val, y_val = get_xy(val_df)
X_test, y_test = get_xy(test_df)

print('usable rows -> train:', len(X_train), 'val:', len(X_val), 'test:', len(X_test))


usable rows -> train: 3214 val: 401 test: 403


In [18]:
baseline_candidates = {
    'tfidf_word_ridge': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=60000)),
        ('reg', Ridge(alpha=3.0, random_state=42)),
    ]),
    'tfidf_char_ridge': Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_features=80000)),
        ('reg', Ridge(alpha=3.0, random_state=42)),
    ]),
}

baseline_rows = []
baseline_models = {}

for model_name, candidate in baseline_candidates.items():
    fitted_model = clone(candidate)
    fitted_model.fit(X_train, y_train)
    baseline_models[model_name] = fitted_model

    for split_name, X_split, y_split in [
        ('val', X_val, y_val),
        ('test', X_test, y_test),
    ]:
        metrics, _ = evaluate_regression_model(fitted_model, model_name, X_split, y_split, split_name)
        baseline_rows.append(metrics)

baseline_metrics_df = pd.DataFrame(baseline_rows).sort_values(['split', 'rmse', 'mae']).reset_index(drop=True)
baseline_metrics_df


,model,split,mae,rmse,r2
0,tfidf_char_ridge,test,441893.437899,1.574751e+06,-0.054701
1,tfidf_word_ridge,test,430802.943322,1.611252e+06,-0.104161
2,tfidf_char_ridge,val,448713.294650,8.493881e+05,-0.568265
3,tfidf_word_ridge,val,435770.221347,1.024017e+06,-1.279403


In [19]:
best_baseline_name = (
    baseline_metrics_df[baseline_metrics_df['split'] == 'val']
    .sort_values(['rmse', 'mae', 'r2'], ascending=[True, True, False])
    .iloc[0]['model']
)
best_baseline_model = baseline_models[best_baseline_name]

val_metrics, val_pred = evaluate_regression_model(best_baseline_model, best_baseline_name, X_val, y_val, 'val')
test_metrics, test_pred = evaluate_regression_model(best_baseline_model, best_baseline_name, X_test, y_test, 'test')

print('selected baseline:', best_baseline_name)
display(pd.DataFrame([val_metrics, test_metrics]))

baseline_preview = pd.DataFrame({
    'clean_text': X_test.head(5),
    'actual_price_tnd': y_test.head(5),
    'predicted_price_tnd': np.round(test_pred[:5], 2),
})
baseline_preview


selected baseline: tfidf_char_ridge


,model,split,mae,rmse,r2
0,tfidf_char_ridge,val,448713.294650,8.493881e+05,-0.568265
1,tfidf_char_ridge,test,441893.437899,1.574751e+06,-0.054701


,clean_text,actual_price_tnd,predicted_price_tnd
0,appartement à louer à ain zaghouen tunis pièce...,1500.0,-57176.11
1,appartement à vendre à sahloul sousse pièces 1...,260000.0,24752.27
2,appartement à louer à le kram tunis pièces 2 c...,1600.0,-132556.83
3,terrain à vendre à monastir surface 301 m2 pri...,195000.0,951478.72
4,terrain à vendre à grand tunis cité mohamed al...,515000.0,733154.47


In [30]:
HF_TOKEN_ENV_VAR = 'HF_TOKEN'
EMBEDDING_MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
EMBEDDING_PREFIX = 'passage: '
EMBED_TRAIN_LIMIT = None


def maybe_limit_split(X, y, limit=None):
    if limit is None or len(X) <= limit:
        return X.reset_index(drop=True), y.reset_index(drop=True)
    return X.iloc[:limit].reset_index(drop=True), y.iloc[:limit].reset_index(drop=True)


def build_hf_client(token_env_var: str = HF_TOKEN_ENV_VAR):
    token = os.getenv(token_env_var)
    if not token:
        raise RuntimeError(
            f'Set the {token_env_var} environment variable before running the multilingual embedding block.'
        )
    return InferenceClient(provider='hf-inference', api_key=token)


def pool_embedding(raw_embedding):
    arr = np.asarray(raw_embedding, dtype=np.float32)
    if arr.ndim == 1:
        vector = arr
    elif arr.ndim == 2:
        vector = arr.mean(axis=0)
    else:
        raise ValueError(f'Unexpected embedding shape: {arr.shape}')
    norm = np.linalg.norm(vector)
    return vector if norm == 0 else vector / norm


def embed_texts_with_hf(client, texts, model_name: str = EMBEDDING_MODEL_ID, prefix: str = EMBEDDING_PREFIX):
    vectors = []
    total = len(texts)
    for idx, text in enumerate(texts, start=1):
        raw_embedding = client.feature_extraction(f'{prefix}{text}', model=model_name)
        vectors.append(pool_embedding(raw_embedding))
        if idx % 100 == 0 or idx == total:
            print(f'embedded {idx}/{total} rows')
    return np.vstack(vectors)


In [37]:
embedding_rows = []
embedding_predictions_df = pd.DataFrame()

try:
    X_train_embed, y_train_embed = maybe_limit_split(X_train, y_train, EMBED_TRAIN_LIMIT)
    try:
        client = build_hf_client()
        print("Client Authentication Successful. Proceeding with embeddings...")
    except Exception as e:
        print(f"Client Authentication Failed because of {e}")

    train_embeddings = embed_texts_with_hf(client, X_train_embed.tolist())
    val_embeddings = embed_texts_with_hf(client, X_val.tolist())
    test_embeddings = embed_texts_with_hf(client, X_test.tolist())

    embedding_model = Ridge(alpha=3.0, random_state=42)
    embedding_model.fit(train_embeddings, y_train_embed)

    val_embed_pred = embedding_model.predict(val_embeddings)
    test_embed_pred = embedding_model.predict(test_embeddings)

    embedding_rows.append(regression_metrics(y_true=y_val, pred=val_embed_pred, model_name='multilingual_e5_ridge', split_name='val'))
    embedding_rows.append(regression_metrics(y_true=y_test, pred=test_embed_pred, model_name='multilingual_e5_ridge', split_name='test'))

    embedding_predictions_df = pd.DataFrame({
        'clean_text': X_test,
        'actual_price_tnd': y_test,
        'predicted_price_tnd': np.round(test_embed_pred, 2),
    })
except Exception as exc:
    embedding_rows.append({
        'model': 'multilingual_e5_ridge',
        'split': 'not_run',
        'status': str(exc),
    })

embedding_metrics_df = pd.DataFrame(embedding_rows)
embedding_metrics_df


Client Authentication Successful. Proceeding with embeddings...


,model,split,status
0,multilingual_e5_ridge,not_run,(Request ID: Root=1-69c55e8b-2232146a673295f84...


In [22]:
models_dir = Path('../artifacts/models')
reports_dir = Path('../artifacts/reports/nlp_description')
models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / f'{best_baseline_name}.pkl'
test_predictions_path = reports_dir / f'{best_baseline_name}_test_predictions.csv'
report_path = reports_dir / 'model1_training_report.json'

with model_path.open('wb') as f:
    pickle.dump(best_baseline_model, f)

pd.DataFrame({
    'clean_text': X_test,
    'actual_price_tnd': y_test,
    'predicted_price_tnd': np.round(test_pred, 2),
}).to_csv(test_predictions_path, index=False)

report = {
    'selected_model': best_baseline_name,
    'model_path': str(model_path),
    'train_rows': int(len(X_train)),
    'val_rows': int(len(X_val)),
    'test_rows': int(len(X_test)),
    'baseline_metrics': baseline_rows,
    'embedding_metrics': embedding_rows,
    'embedding_model_id': EMBEDDING_MODEL_ID,
    'embedding_train_limit': EMBED_TRAIN_LIMIT,
    'notes': [
        'The multilingual E5 block is optional and requires HF_TOKEN in the notebook environment.',
        'The saved pickle contains the best local TF-IDF baseline, not the remote embedding client.',
    ],
}

report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('saved model ->', model_path)
print('saved predictions ->', test_predictions_path)
print('saved report ->', report_path)


saved model -> ..\artifacts\models\tfidf_char_ridge.pkl
saved predictions -> ..\artifacts\reports\nlp_description\tfidf_char_ridge_test_predictions.csv
saved report -> ..\artifacts\reports\nlp_description\model1_training_report.json
